# Approaching the match between _context_ in Strucke and _features_ in SEAD

In [14]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)


## Load c14_master_v08.xlsx


In [15]:
excel_path = "../data/c14_master_v08.xlsx"
df = pd.read_excel(excel_path)
df[['context_id', 'context_type']].head()


,context_id,context_type
0,A1328,Kokgrop
1,A4,Stenpackning
2,A6,Stenpackning
3,A8,Härd
4,A23,Grop


### context_id and context_type overview


In [16]:
print(f"{df['context_id'].nunique()} unique context_id values")
print(f"{df['context_type'].nunique()} unique context_type values")
df['context_type'].value_counts().head(20)


10503 unique context_id values
1053 unique context_type values


context_type
Härd                  6966
Stolphål              5114
Grop                  1822
Lager                 1648
Röjningsröse          1105
Kokgrop               1098
Kulturlager            886
Härdgrop               478
Stensättning           466
Skelettgrav            397
Brunn                  395
Ugn                    352
Ränna                  277
Stensättning, rund     267
Grophus                232
Nedgrävning            227
Skärvstenshög          215
Kolningsgrop           197
Slaggvarp              195
Okänd                  195
Name: count, dtype: int64

## Connect to sead_staging database


In [17]:
import os

from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

DB_HOST = os.environ["DB_HOST"]
DB_PORT = os.environ["DB_PORT"]
DB_NAME = os.environ["DB_NAME"]
DB_USER = os.environ["DB_USER"]
DB_PASSWORD = os.environ["DB_PASSWORD"]

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")


## Load tbl_features and tbl_feature_types


In [18]:
features = pd.read_sql('select * from public.tbl_features order by feature_id', engine)
print(f'{len(features)} rows in tbl_features')
features.head()


2669 rows in tbl_features


,feature_id,feature_type_id,feature_name,feature_description,date_updated
0,1,26,5854,NaN,2013-05-13 11:29:56.070308+02:00
1,2,26,16614,NaN,2013-05-13 11:29:56.070308+02:00
2,3,7,11196,NaN,2013-05-13 11:29:56.070308+02:00
3,4,8,2015,NaN,2013-05-13 11:29:56.070308+02:00
4,5,7,2886,NaN,2013-05-13 11:29:56.070308+02:00


In [19]:
feature_types = pd.read_sql('select * from public.tbl_feature_types order by feature_type_id', engine)
print(f'{len(feature_types)} rows in tbl_feature_types')
feature_types.head()


156 rows in tbl_feature_types


,feature_type_id,feature_type_name,feature_type_description,date_updated
0,1,Building,Structure interpreted as having an enclosed sp...,2012-09-21 18:51:47.967181+02:00
1,2,Hunting pit,Pit interpreted as an animal trap,2012-09-21 18:51:47.967181+02:00
2,4,Fortification,"Defensive structure, e.g. wall, earthwork, tow...",2012-10-11 15:37:42.275139+02:00
3,5,Layer,Any constrained sedimentological unit,2012-10-11 15:50:48.349888+02:00
4,6,Grave/burial,Burial of undefined type,2012-11-01 14:51:37.977054+01:00


## Join tbl_features with tbl_feature_types on feature_type_id


In [20]:
features_joined = features.merge(feature_types, on='feature_type_id', how='left')
features_joined[['feature_id', 'feature_name', 'feature_type_name']]


,feature_id,feature_name,feature_type_name
0,1,5854,Post hole
1,2,16614,Post hole
2,3,11196,Pit
3,4,2015,Cooking pit
4,5,2886,Pit
...,...,...,...
2664,4815,Feature,Settlement site
2665,4816,Feature,Settlement site
2666,4817,Feature,Settlement site
2667,4818,Feature,Settlement site


## Compare feature_name (tbl_features join tbl_feature_types) with context_id (xlsx)


In [21]:
feature_names = set(features_joined['feature_name'].dropna())
context_ids = set(df['context_id'].dropna())

common = feature_names & context_ids
print(f'{len(feature_names)} distinct feature_name values')
print(f'{len(context_ids)} distinct context_id values')
print(f'{len(common)} values appear in both feature_name and context_id')


2078 distinct feature_name values
10503 distinct context_id values
509 values appear in both feature_name and context_id


### Rows in the joined features table matching a context_id


In [22]:
features_joined[features_joined['feature_name'].isin(common)].sort_values('feature_name').head(20)


,feature_id,feature_type_id,feature_name,feature_description,date_updated_x,feature_type_name,feature_type_description,date_updated_y
2280,4431,551,0,NaN,2020-01-09 09:37:49.705454+01:00,Settlement site,A site previously inhabited where worked objec...,2019-12-20 14:45:52.481448+01:00
2281,4432,565,0,NaN,2020-01-09 09:37:49.705454+01:00,Unprocessed clay,NaN,2019-12-20 14:45:52.481448+01:00
2275,4426,6,0,NaN,2020-01-09 09:37:49.705454+01:00,Grave/burial,Burial of undefined type,2012-11-01 14:51:37.977054+01:00
1553,1643,6,1,NaN,2014-02-19 15:28:44.312000+01:00,Grave/burial,Burial of undefined type,2012-11-01 14:51:37.977054+01:00
1539,1629,27,1,NaN,2014-02-19 15:28:44.312000+01:00,Undefined,No feature description available,2013-04-16 16:45:51.284320+02:00
2139,4290,6,1,NaN,2020-01-09 09:37:49.705454+01:00,Grave/burial,Burial of undefined type,2012-11-01 14:51:37.977054+01:00
2059,4210,550,1,NaN,2020-01-09 09:37:49.705454+01:00,Unknown,Feature type is either of a unknown character ...,2025-05-28 10:11:14.301236+02:00
1457,1547,6,1,NaN,2014-02-19 15:28:44.312000+01:00,Grave/burial,Burial of undefined type,2012-11-01 14:51:37.977054+01:00
1602,1692,19,1,NaN,2014-02-19 15:28:44.312000+01:00,Hearth,Fireplace,2012-11-01 15:28:41.090057+01:00
1610,1700,19,1,NaN,2014-02-19 15:28:44.312000+01:00,Hearth,Fireplace,2012-11-01 15:28:41.090057+01:00


### Corresponding rows in the xlsx for the matched context_id values


In [23]:
df[df['context_id'].isin(common)][['context_id', 'context_type']].drop_duplicates().sort_values('context_id')


,context_id,context_type
6509,0,Störhål
2109,0,Grop
27985,0,Härd
6695,0,okänd
22276,1,Odlingsterrass
...,...,...
26275,S1,Kollager
1317,S21,Lager
29576,S4,Lager
27894,XII,Grophus


## Inner join context_id (xlsx) with feature_name (joined features table)


In [24]:
context_features = df[['context_id', 'context_type']].merge(
    features_joined,
    how='inner',
    left_on='context_id',
    right_on='feature_name',
)
print(f'{len(context_features)} rows in the joined table')
context_features = context_features[['feature_id', 'context_id', 'feature_name', 'context_type', 'feature_type_name', 'feature_description', 'feature_type_description', 'feature_type_id', 'date_updated_x', 'date_updated_y']]
context_features 


25444 rows in the joined table


,feature_id,context_id,feature_name,context_type,feature_type_name,feature_description,feature_type_description,feature_type_id,date_updated_x,date_updated_y
0,919,A4,A4,Stenpackning,Layer,Cultural layer\n,Any constrained sedimentological unit,5,2013-11-13 15:33:32.218000+01:00,2012-10-11 15:50:48.349888+02:00
1,944,A4,A4,Stenpackning,Undefined,NaN,No feature description available,27,2013-11-13 15:33:32.218000+01:00,2013-04-16 16:45:51.284320+02:00
2,1073,A4,A4,Stenpackning,Hearth,NaN,Fireplace,19,2013-11-13 15:33:32.218000+01:00,2012-11-01 15:28:41.090057+01:00
3,1430,A4,A4,Stenpackning,Undefined,Aggregation of stones,No feature description available,27,2013-11-13 15:33:32.218000+01:00,2013-04-16 16:45:51.284320+02:00
4,4086,A4,A4,Stenpackning,City layer,NaN,A cultural layer of a city character.,567,2020-01-09 09:37:49.705454+01:00,2019-12-20 14:45:52.481448+01:00
...,...,...,...,...,...,...,...,...,...,...
25439,1421,A23,A23,Lager,Pit,NaN,Natural or man made hole or depression of unsp...,7,2013-11-13 15:33:32.218000+01:00,2012-11-01 14:55:28.292677+01:00
25440,1467,A23,A23,Lager,Undefined,NaN,No feature description available,27,2013-11-13 15:33:32.218000+01:00,2013-04-16 16:45:51.284320+02:00
25441,901,A14,A14,"Stensättning, rektangulär",Pit,NaN,Natural or man made hole or depression of unsp...,7,2013-11-13 15:33:32.218000+01:00,2012-11-01 14:55:28.292677+01:00
25442,1098,A14,A14,"Stensättning, rektangulär",Hearth,NaN,Fireplace,19,2013-11-13 15:33:32.218000+01:00,2012-11-01 15:28:41.090057+01:00


## Another approach to understanding the column _"context_type"_

Instead of looking for matches directly based on context_id and feature_name, we will look at values from the feature_type_name and the unique values of context_type to check with AI for translations and matches

In [25]:
context_types = sorted(df['context_type'].dropna().unique())
print(f'{len(context_types)} unique context_type values')
context_types


1053 unique context_type values


["'Härd",
 '/Stolphål',
 '12338',
 '12404',
 '1253',
 '242',
 'A1',
 'A623',
 'Aktivitetslager',
 'Aktivitetsyta',
 'Ankare',
 'Anläggning',
 'Annan anläggning',
 'Ansamling av tra',
 'Arbetsgrop',
 'Arbetsyta',
 'Asklager',
 'Atenläggning',
 'Atolphål',
 'Atörhål',
 'Avallsgrop',
 'Avfalls/skärvstenslager',
 'Avfallsanläggning',
 'Avfallsbinge',
 'Avfallsdeponi',
 'Avfallsgrop',
 'Avfallsgrop/härd',
 'Avfallshög',
 'Avfallslager',
 'Avfallslagr',
 'Avsättningslager',
 'Bakugn',
 'Barksjok',
 'Barngrav',
 'Bearbetat trä',
 'Bendeposition',
 'Bendepå',
 'Bengrop',
 'Bengömma',
 'Benkocentration',
 'Benkoncentration',
 'Benlager',
 'Bensamling',
 'Beredningsgrop',
 'Bjälklag',
 'Björk',
 'Björngrav',
 'Blockgrav',
 'Blockgrav, oval',
 'Blockkonstruktion/stensättning',
 'Blästerugn',
 'Blästplats',
 'Blästugn',
 'Bogårdsmur',
 'Boplatsgrop',
 'Boplatslämning övrig',
 'Boplatsvall',
 'Bordläggning',
 'Bordplanka',
 'Bordstake',
 'Borgvall',
 'Bottenlager',
 'Bottensediment',
 'Bottenstock'

In [26]:
feature_type_names = sorted(feature_types['feature_type_name'].dropna().unique())
print(f'{len(feature_type_names)} unique feature_type_name values')
feature_type_names


155 unique feature_type_name values


['Abutment',
 'Barrel',
 'Barrier',
 'Beam',
 'Bedding layer',
 'Border marker',
 'Box',
 'Brick floor',
 'Brick kiln',
 'Bridge',
 'Building',
 'Carriage',
 'Castle',
 'Cellar',
 'Cesspit',
 'Charcoal kiln',
 'Church',
 'Cist grave',
 'City layer',
 'Coffin',
 'Collection pit (tar)',
 'Container',
 'Cooking pit',
 'Crafts site',
 'Cult site',
 'Cultural layer',
 'Dam',
 'Dark Earth',
 'Ditch',
 'Dock',
 'Doorsill timber',
 'Dragare (translation pending)',
 'Drain',
 'Drawbridge',
 'Enclosure',
 'Excavation area',
 'Excavation unit/context/fill',
 'Fill',
 'Fill layer',
 'Floor',
 'Floor joist',
 'Floor layer',
 'Flooring',
 'Fortification',
 'Fortified tower',
 'Foundation',
 'Furnace',
 'Gabion',
 'Grave/burial',
 'Grave/burial and settlement site',
 'Grave/burial?',
 'Grillage ',
 'Ground gutter',
 'Gutter',
 'Heap of fire-cracked stones',
 'Hearth',
 'Hoard',
 'Horse gin',
 'Hunting pit',
 'Industrial deposits',
 'Iron production site',
 'Jetty/Quay',
 'Kiln',
 'Layer',
 'Log coffi

### Manual translation of the most common context_type values

`context_type` has 1053 unique values, many of them rare misspellings or one-off variants. The top ~100 values already cover about 91% of all rows, so the dictionary below focuses on translating those higher-frequency Swedish terms into English, and only where the translation plausibly corresponds to an existing `feature_type_name`. It is not an exhaustive translation of every value.


In [27]:
context_type_translation = {
    'Härd': 'Hearth',
    'härd': 'Hearth',
    'Stolphål': 'Post hole',
    'Grop': 'Pit',
    'Lager': 'Layer',
    'Kokgrop': 'Cooking pit',
    'Kulturlager': 'Cultural layer',
    'Stensättning': 'Stone setting',
    'Brunn': 'Well',
    'Ugn': 'Oven',
    'Ränna': 'Gutter',
    'Skärvstenshög': 'Heap of fire-cracked stones',
    'Okänd': 'Unknown',
    'okänd': 'Unknown',
    'Fångstgrop': 'Hunting pit',
    'Golvlager': 'Floor layer',
    'Påle': 'Pole',
    'Dike': 'Ditch',
    'Väggränna': 'Wall trench',
    'Hällkista': 'Cist grave',
    'Förrådsgrop': 'Storage pit',
    'Tegelugn': 'Brick kiln',
    'Golv': 'Floor',
    'Mur': 'Wall',
    'Väg': 'Road',
    'Kallrost': 'Roast bed',
    'Stolpe': 'Post',
    'Trägolv': 'Wooden floor',
    'Störhål': 'Stake hole',
    'Utfyllnadslager': 'Fill layer',
    'Stockbåt': 'Logboat',
    'Vrak': 'Shipwreck',
}

feature_type_lookup = {name.strip().lower(): name for name in feature_type_names}

df['context_type_en'] = df['context_type'].map(context_type_translation)

translated = df['context_type_en'].notna()
direct_match = df['context_type_en'].str.lower().isin(feature_type_lookup)

print(f'{translated.sum()} rows have a manual translation')
print(f'{direct_match.sum()} rows have a translated context_type that directly matches a feature_type_name')
print(f'{df.loc[direct_match, "context_type"].nunique()} distinct context_type values have a direct English match')


20170 rows have a manual translation
20170 rows have a translated context_type that directly matches a feature_type_name
32 distinct context_type values have a direct English match


In [28]:
df.loc[direct_match, ['context_type', 'context_type_en']].drop_duplicates().sort_values('context_type')


,context_type,context_type_en
302,Brunn,Well
1160,Dike,Ditch
1580,Fångstgrop,Hunting pit
843,Förrådsgrop,Storage pit
10955,Golv,Floor
721,Golvlager,Floor layer
4,Grop,Pit
513,Hällkista,Cist grave
3,Härd,Hearth
2681,Kallrost,Roast bed


## Unmatched context_type values


In [29]:
unmatched_mask = df['context_type'].notna() & ~direct_match
unmatched_counts = (
    df.loc[unmatched_mask, 'context_type']
    .value_counts()
    .rename_axis('context_type')
    .reset_index(name='count')
)
print(f'{len(unmatched_counts)} distinct unmatched context_type values, {unmatched_counts["count"].sum()} rows')
unmatched_counts


1021 distinct unmatched context_type values, 9625 rows


,context_type,count
0,Röjningsröse,1105
1,Härdgrop,478
2,Skelettgrav,397
3,"Stensättning, rund",267
4,Grophus,232
...,...,...
1016,Bordläggning,1
1017,Grophau,1
1018,Båtstöd,1
1019,Kokgrop/ugn,1


In [30]:
unmatched_counts.to_csv('../output/context_type_unmatched.csv', index=False)
